# 01 — Data Understanding

## tl;dr

This notebook inventories and profiles the actual Kaggle files available locally. It establishes the grain, date coverage, candidate keys, missingness, duplicates, memory use, and descriptive context needed before cleaning or analysis. Conclusions here are deliberately preliminary.

## Context & Methods

**Decision supported:** determine whether the source data is structurally suitable for retail demand and operations analysis.

### Key assumptions

- `sales` is observed sales, not unconstrained customer demand.
- A zero-sales row may reflect no demand, closure, or unavailable stock; the dataset cannot distinguish these directly.
- Candidate keys are validated empirically and are not treated as guaranteed database constraints.
- The Kaggle files in `data/raw/` are the controlling source for this notebook.

In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
# Resolve the repository root dynamically so the notebook works from either
# the project root or the notebooks directory without a local absolute path.
candidate = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (candidate, *candidate.parents)
    if (path / "data").is_dir() and (path / "src").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DATA_DIR = PROJECT_ROOT / "outputs" / "processed_data"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
for directory in (PROCESSED_DATA_DIR, OUTPUT_DATA_DIR, FIGURE_DIR, TABLE_DIR):
    # Creating output folders is safe; raw source files are never modified.
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

In [2]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.data_loader import (
    CANDIDATE_KEYS,
    DATASET_DESCRIPTIONS,
    EXPECTED_FILES,
    inspect_available_files,
    load_all_available,
    profile_dataset,
)

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## Data

### 1. Verify available inputs

**Why this step is needed:** A reproducible analysis must confirm which files are actually available before referring to them. This prevents later failures and avoids silently analyzing an incomplete dataset.

In [3]:
inventory = inspect_available_files(RAW_DATA_DIR)
display(inventory)
missing_files = inventory.loc[~inventory["available"], "file"].tolist()
if missing_files:
    raise FileNotFoundError(f"Missing expected files: {missing_files}. See data/README.md.")
print(f"Verified {inventory['available'].sum()} of {len(EXPECTED_FILES)} expected files.")

,file,available,size_mb
0,train.csv,True,116.158
1,test.csv,True,0.975
2,stores.csv,True,0.001
3,transactions.csv,True,1.481
4,holidays_events.csv,True,0.021
5,oil.csv,True,0.020
6,sample_submission.csv,True,0.326


Verified 7 of 7 expected files.


### 2. Load the available CSVs

**Why this step is needed:** Loading through one reusable function gives every notebook consistent date parsing, file checks, and error messages.

In [4]:
datasets = load_all_available(RAW_DATA_DIR)
profiles = pd.DataFrame(
    [profile_dataset(filename, frame) for filename, frame in datasets.items()]
)
display(profiles)

,file,rows,columns,memory_mb,exact_duplicate_rows,missing_cells,date_min,date_max
0,train.csv,3000888,6,308.390,0,0,2013-01-01,2017-08-15
1,test.csv,28512,5,2.710,0,0,2017-08-16,2017-08-31
2,stores.csv,54,5,0.010,0,0,NaT,NaT
3,transactions.csv,83488,3,1.910,0,0,2013-01-01,2017-08-15
4,holidays_events.csv,350,6,0.090,0,0,2012-03-02,2017-12-26
5,oil.csv,1218,2,0.020,0,43,2013-01-01,2017-08-31
6,sample_submission.csv,28512,2,0.440,0,0,NaT,NaT


**How to interpret this step:** row and column counts confirm the scale of each source, while memory usage indicates whether full-population analysis is practical. The tables are loaded independently here so that their grain is understood before any merge can silently multiply or remove records.

### 3. Explain the datasets and their proposed grain

**Why this step is needed:** The grain defines what one row represents. Understanding it before merging is essential because incompatible grains can duplicate sales or remove valid records.

In [5]:
grain_map = {
    "train.csv": "one date × store × product family",
    "test.csv": "one future date × store × product family",
    "stores.csv": "one store",
    "transactions.csv": "one date × store with a reported transaction count",
    "holidays_events.csv": "one holiday/event record; dates may repeat across events/scopes",
    "oil.csv": "one reported date",
    "sample_submission.csv": "one test-row id",
}
dataset_dictionary = pd.DataFrame(
    {
        "file": list(DATASET_DESCRIPTIONS),
        "represents": list(DATASET_DESCRIPTIONS.values()),
        "proposed_grain": [grain_map[name] for name in DATASET_DESCRIPTIONS],
        "candidate_key": [", ".join(CANDIDATE_KEYS[name]) for name in DATASET_DESCRIPTIONS],
    }
)
display(dataset_dictionary)

,file,represents,proposed_grain,candidate_key
0,train.csv,Historical daily sales at date-store-product-f...,one date × store × product family,id
1,test.csv,Competition forecast horizon at date-store-pro...,one future date × store × product family,id
2,stores.csv,"Store location, type, and cluster attributes.",one store,store_nbr
3,transactions.csv,Observed daily transaction counts by store.,one date × store with a reported transaction c...,"date, store_nbr"
4,holidays_events.csv,"Holiday and event records at national, regiona...",one holiday/event record; dates may repeat acr...,"date, type, locale, locale_name, description, ..."
5,oil.csv,Daily WTI oil-price observations supplied by t...,one reported date,date
6,sample_submission.csv,Required Kaggle prediction-output structure.,one test-row id,id


## Results

### 4. Columns and data types

**Why this step is needed:** Schema checks confirm which fields are numeric, categorical, or temporal and reveal type corrections needed before calculation or joining.

In [6]:
schema_rows = []
for filename, frame in datasets.items():
    for column, dtype in frame.dtypes.items():
        schema_rows.append({"file": filename, "column": column, "dtype": str(dtype)})
schema = pd.DataFrame(schema_rows)
display(schema)

,file,column,dtype
0,train.csv,id,int64
1,train.csv,date,datetime64[ns]
2,train.csv,store_nbr,int64
3,train.csv,family,object
4,train.csv,sales,float64
5,train.csv,onpromotion,int64
6,test.csv,id,int64
7,test.csv,date,datetime64[ns]
8,test.csv,store_nbr,int64
9,test.csv,family,object


### 5. Date coverage, stores, and product families

**Why this step is needed:** Coverage boundaries and entity counts establish the population represented by the analysis and expose gaps or unexpected scope differences.

In [7]:
date_coverage = []
for filename, frame in datasets.items():
    if "date" in frame:
        date_coverage.append(
            {
                "file": filename,
                "start_date": frame["date"].min().date(),
                "end_date": frame["date"].max().date(),
                "unique_dates": frame["date"].nunique(),
                "invalid_dates": int(frame["date"].isna().sum()),
            }
        )
display(pd.DataFrame(date_coverage))

train = datasets["train.csv"]
stores = datasets["stores.csv"]
print(f"Training stores: {train['store_nbr'].nunique():,}")
print(f"Training product families: {train['family'].nunique():,}")
print(f"Store master rows: {len(stores):,}; unique store numbers: {stores['store_nbr'].nunique():,}")
display(pd.DataFrame({"product_family": sorted(train["family"].unique())}))

,file,start_date,end_date,unique_dates,invalid_dates
0,train.csv,2013-01-01,2017-08-15,1684,0
1,test.csv,2017-08-16,2017-08-31,16,0
2,transactions.csv,2013-01-01,2017-08-15,1682,0
3,holidays_events.csv,2012-03-02,2017-12-26,312,0
4,oil.csv,2013-01-01,2017-08-31,1218,0


Training stores: 54
Training product families: 33
Store master rows: 54; unique store numbers: 54


,product_family
0,AUTOMOTIVE
1,BABY CARE
2,BEAUTY
3,BEVERAGES
4,BOOKS
5,BREAD/BAKERY
6,CELEBRATION
7,CLEANING
8,DAIRY
9,DELI


### 6. Candidate-key, duplicate, and join checks

**Why this step is needed:** Keys protect the intended unit of analysis. Duplicate keys can double-count demand, while unmatched store keys can invalidate store-level comparisons.

In [8]:
key_checks = []
for filename, key in CANDIDATE_KEYS.items():
    frame = datasets[filename]
    key_checks.append(
        {
            "file": filename,
            "candidate_key": ", ".join(key),
            "duplicate_key_rows": int(frame.duplicated(key, keep=False).sum()),
            "exact_duplicate_rows": int(frame.duplicated().sum()),
        }
    )
display(pd.DataFrame(key_checks))

sales_key_duplicates = train.duplicated(["date", "store_nbr", "family"], keep=False).sum()
orphan_train_stores = len(set(train["store_nbr"]) - set(stores["store_nbr"]))
orphan_test_stores = len(set(datasets["test.csv"]["store_nbr"]) - set(stores["store_nbr"]))
print(f"Duplicate train date-store-family rows: {sales_key_duplicates:,}")
print(f"Training store keys absent from stores.csv: {orphan_train_stores:,}")
print(f"Test store keys absent from stores.csv: {orphan_test_stores:,}")

,file,candidate_key,duplicate_key_rows,exact_duplicate_rows
0,train.csv,id,0,0
1,test.csv,id,0,0
2,stores.csv,store_nbr,0,0
3,transactions.csv,"date, store_nbr",0,0
4,holidays_events.csv,"date, type, locale, locale_name, description, ...",0,0
5,oil.csv,date,0,0
6,sample_submission.csv,id,0,0


Duplicate train date-store-family rows: 0
Training store keys absent from stores.csv: 0
Test store keys absent from stores.csv: 0


**Why this matters operationally:** a demand-planning result is only credible if each sales observation represents one intended planning unit. Duplicate keys could double-count demand, while unmatched stores would remove geography, type, and cluster context from store comparisons.

### 7. Missing values and memory usage

**Why this step is needed:** Missingness affects which analyses are trustworthy, while memory usage helps decide whether to work with the full panel or a justified subset.

In [9]:
missing_rows = []
for filename, frame in datasets.items():
    for column in frame.columns:
        count = int(frame[column].isna().sum())
        missing_rows.append(
            {
                "file": filename,
                "column": column,
                "missing_count": count,
                "missing_pct": 100 * count / len(frame),
            }
        )
missing_summary = pd.DataFrame(missing_rows)
display(missing_summary.loc[missing_summary["missing_count"].gt(0)])
display(profiles[["file", "rows", "columns", "memory_mb"]])

,file,column,missing_count,missing_pct
26,oil.csv,dcoilwtico,43,3.530


,file,rows,columns,memory_mb
0,train.csv,3000888,6,308.390
1,test.csv,28512,5,2.710
2,stores.csv,54,5,0.010
3,transactions.csv,83488,3,1.910
4,holidays_events.csv,350,6,0.090
5,oil.csv,1218,2,0.020
6,sample_submission.csv,28512,2,0.440


### 8. Descriptive statistics

**Why this step is needed:** Basic distributions provide a first reasonableness check for ranges, scales, skew, and unusual values before making business interpretations.

In [10]:
for filename in ("train.csv", "transactions.csv", "oil.csv", "stores.csv"):
    print()
    print(filename)
    display(datasets[filename].describe(include="all").T)


train.csv


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
id,"3,000,888.000",NaN,NaN,NaN,"1,500,443.500",0.000,"750,221.750","1,500,443.500","2,250,665.250","3,000,887.000","866,281.892"
date,3000888,NaN,NaN,NaN,2015-04-24 08:27:04.703088384,2013-01-01 00:00:00,2014-02-26 18:00:00,2015-04-24 12:00:00,2016-06-19 06:00:00,2017-08-15 00:00:00,NaN
store_nbr,"3,000,888.000",NaN,NaN,NaN,27.500,1.000,14.000,27.500,41.000,54.000,15.586
family,3000888,33,AUTOMOTIVE,90936,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sales,"3,000,888.000",NaN,NaN,NaN,357.776,0.000,0.000,11.000,195.847,"124,717.000","1,101.998"
onpromotion,"3,000,888.000",NaN,NaN,NaN,2.603,0.000,0.000,0.000,0.000,741.000,12.219



transactions.csv


,count,mean,min,25%,50%,75%,max,std
date,83488,2015-05-20 16:07:40.866232064,2013-01-01 00:00:00,2014-03-27 00:00:00,2015-06-08 00:00:00,2016-07-14 06:00:00,2017-08-15 00:00:00,NaN
store_nbr,"83,488.000",26.939,1.000,13.000,27.000,40.000,54.000,15.608
transactions,"83,488.000","1,694.602",5.000,"1,046.000","1,393.000","2,079.000","8,359.000",963.287



oil.csv


,count,mean,min,25%,50%,75%,max,std
date,1218,2015-05-02 12:00:00,2013-01-01 00:00:00,2014-03-03 06:00:00,2015-05-02 12:00:00,2016-06-30 18:00:00,2017-08-31 00:00:00,NaN
dcoilwtico,"1,175.000",67.714,26.190,46.405,53.190,95.660,110.620,25.630



stores.csv


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
store_nbr,54.000,NaN,NaN,NaN,27.500,15.732,1.000,14.250,27.500,40.750,54.000
city,54,22,Quito,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,54,16,Pichincha,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,54,5,D,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cluster,54.000,NaN,NaN,NaN,8.481,4.693,1.000,4.000,8.500,13.000,17.000


### 9. Domain-specific first checks

**Why this step is needed:** Retail data requires checks that generic profiling misses, such as negative demand, zero sales, unusual promotions, missing oil prices, and overlapping event dates.

In [11]:
holidays = datasets["holidays_events.csv"]
transactions = datasets["transactions.csv"]
oil = datasets["oil.csv"]
domain_checks = pd.DataFrame(
    [
        {"check": "negative sales rows", "value": int(train["sales"].lt(0).sum())},
        {"check": "zero sales rows", "value": int(train["sales"].eq(0).sum())},
        {"check": "negative promotion rows", "value": int(train["onpromotion"].lt(0).sum())},
        {"check": "negative transaction rows", "value": int(transactions["transactions"].lt(0).sum())},
        {"check": "oil rows with missing price", "value": int(oil["dcoilwtico"].isna().sum())},
        {"check": "additional holiday rows on repeated dates", "value": int(holidays.duplicated("date").sum())},
        {"check": "transferred holiday records", "value": int(holidays["transferred"].sum())},
    ]
)
display(domain_checks)

,check,value
0,negative sales rows,0
1,zero sales rows,939130
2,negative promotion rows,0
3,negative transaction rows,0
4,oil rows with missing price,43
5,additional holiday rows on repeated dates,38
6,transferred holiday records,12


**Why values are flagged rather than deleted:** retail extremes may be genuine operational signals. Zero sales can reflect low demand, closures, or stockouts; spikes can reflect promotions, holidays, paydays, or disruptions. Notebook 02 investigates these observations in context before applying any treatment.

### 10. Small data dictionary

**Why this step is needed:** A shared definition of each important field keeps the technical work and the MBA business narrative aligned.

In [12]:
data_dictionary = pd.DataFrame(
    [
        ("date", "Calendar date", "Shared time join; daily planning grain"),
        ("store_nbr", "Store identifier", "Joins sales/test/transactions to stores"),
        ("family", "Product-family category", "Category-level demand dimension"),
        ("sales", "Observed sales for a store-family-date", "Forecast target; may include fractional units"),
        ("onpromotion", "Number of promoted items in a family", "Promotion-intensity indicator, not causal assignment"),
        ("city / state", "Store geography", "Used for geographic and holiday-scope matching"),
        ("type", "Anonymized store type", "Store-format comparison dimension"),
        ("cluster", "Grouping of similar stores", "Store-segmentation dimension"),
        ("transactions", "Reported daily store transactions", "Traffic proxy at store-date grain"),
        ("holiday fields", "Event type, locale, description, transfer flag", "Context requiring scope-aware joins"),
        ("dcoilwtico", "WTI oil price", "Exploratory macro-context variable"),
    ],
    columns=["field", "meaning", "analytical_role"],
)
display(data_dictionary)

,field,meaning,analytical_role
0,date,Calendar date,Shared time join; daily planning grain
1,store_nbr,Store identifier,Joins sales/test/transactions to stores
2,family,Product-family category,Category-level demand dimension
3,sales,Observed sales for a store-family-date,Forecast target; may include fractional units
4,onpromotion,Number of promoted items in a family,"Promotion-intensity indicator, not causal assi..."
5,city / state,Store geography,Used for geographic and holiday-scope matching
6,type,Anonymized store type,Store-format comparison dimension
7,cluster,Grouping of similar stores,Store-segmentation dimension
8,transactions,Reported daily store transactions,Traffic proxy at store-date grain
9,holiday fields,"Event type, locale, description, transfer flag",Context requiring scope-aware joins


### 11. Export small audit tables for report and presentation use

**Why this step is needed:** Compact evidence tables make important checks reusable in the final report and slides without copying large raw outputs.

In [13]:
profiles.to_csv(TABLE_DIR / "phase1_dataset_profile.csv", index=False)
pd.DataFrame(date_coverage).to_csv(TABLE_DIR / "phase1_date_coverage.csv", index=False)
pd.DataFrame(key_checks).to_csv(TABLE_DIR / "phase1_key_checks.csv", index=False)
domain_checks.to_csv(TABLE_DIR / "phase1_domain_checks.csv", index=False)
data_dictionary.to_csv(TABLE_DIR / "phase1_data_dictionary.csv", index=False)
print("Saved five compact Phase 1 audit tables to reports/tables/.")

Saved five compact Phase 1 audit tables to reports/tables/.


## Initial Data Observations

In [14]:
expected_panel_rows = train["date"].nunique() * train["store_nbr"].nunique() * train["family"].nunique()
zero_rate = 100 * train["sales"].eq(0).mean()
oil_missing_rate = 100 * oil["dcoilwtico"].isna().mean()
display(Markdown(f'''
- All seven expected competition files are present locally.
- The training table has **{len(train):,} rows** from **{train['date'].min().date()} to {train['date'].max().date()}**, covering **{train['store_nbr'].nunique()} stores** and **{train['family'].nunique()} product families**.
- Its row count equals the complete date × store × family grid (**{expected_panel_rows:,}**), and no duplicate rows occur at that grain.
- There are no negative sales values, but **{train['sales'].eq(0).sum():,} rows ({zero_rate:.2f}%)** have zero sales. These require contextual review rather than automatic deletion.
- Oil price is the only source column with explicit nulls: **{oil['dcoilwtico'].isna().sum():,} rows ({oil_missing_rate:.2f}% of reported oil dates)**.
- Transaction records cover **{transactions['date'].nunique():,} dates** and are not a complete store-date grid; absence must be investigated before imputation.
- Holiday dates are not unique because several events or scopes can occur on the same date. These records must be aggregated or matched by national/regional/local scope before joining to sales.
- These are structural observations, not evidence of operational causes or performance differences.
'''))


- All seven expected competition files are present locally.
- The training table has **3,000,888 rows** from **2013-01-01 to 2017-08-15**, covering **54 stores** and **33 product families**.
- Its row count equals the complete date × store × family grid (**3,000,888**), and no duplicate rows occur at that grain.
- There are no negative sales values, but **939,130 rows (31.30%)** have zero sales. These require contextual review rather than automatic deletion.
- Oil price is the only source column with explicit nulls: **43 rows (3.53% of reported oil dates)**.
- Transaction records cover **1,682 dates** and are not a complete store-date grid; absence must be investigated before imputation.
- Holiday dates are not unique because several events or scopes can occur on the same date. These records must be aggregated or matched by national/regional/local scope before joining to sales.
- These are structural observations, not evidence of operational causes or performance differences.


## Takeaways

- The core sales table is structurally complete at its stated grain and can support comparative analysis.
- Context tables cannot be merged naively: transactions are incomplete, oil needs time-aware treatment, and holiday scope can create join multiplication.
- Zero sales should be investigated, not deleted, because the dataset cannot distinguish low demand, closures, and stockouts.
- Notebook 02 will document cleaning choices, validate each merge, and select a detailed-analysis scope from observed volume, variability, and completeness.

## PPT-ready Phase 1 message

**Slide headline:** The source data is structurally suitable for demand analysis, but contextual variables require careful preparation.

**Evidence to show:**

- 3.00 million daily store-family observations covering 54 stores and 33 product families.
- The sales panel is complete and unique at the date × store × family grain.
- 31.30% of rows report zero sales; these are retained for investigation rather than treated as errors.
- Oil-price gaps, incomplete transaction coverage, and overlapping holiday records require controlled, validated joins.

**Management interpretation:** the dataset can support demand-pattern and store/category comparisons, but it cannot directly reveal stockouts or optimize inventory because stock-on-hand, lead-time, service-level, cost, and margin data are absent.

**Do not claim yet:** promotion causality, holiday impact, store underperformance, or forecast improvement. Those require the later analytical phases.